**What is the average number of orders per customer? Are there high-value repeat customers?**


In [0]:
%sql
use catalog workspace;

use schema northwind_sales_analysis;

In [0]:
%sql
-- Average number of orders per customer
SELECT 
  AVG(order_count) AS avg_orders_per_customer
FROM (
  SELECT 
    customerid, 
    COUNT(orderid) AS order_count
  FROM orders
  GROUP BY customerid
);



avg_orders_per_customer
8.645833333333334


In [0]:
%sql
-- High-value repeat customers (e.g., customers with >2 orders and total sales > $10,000)
SELECT 
  o.customerid,
  COUNT(od.orderid) AS total_orders,
  ROUND(SUM(od.total_price),2) AS total_sales
FROM orders o
JOIN orders_details od ON o.orderid = od.orderid
GROUP BY o.customerid
HAVING COUNT(od.orderid) > 2 AND SUM(od.total_price) > 10000
ORDER BY total_sales DESC;

customerid,total_orders,total_sales
QUICK,86,117477.44
SAVEA,116,115663.79
ERNSH,102,113229.73
HUNGO,55,57311.14
RATTC,71,52243.41
HANAR,32,34099.0
FOLKO,45,32551.7
MEREP,32,32202.05
KOENE,39,31744.2
QUEEN,40,30222.5


**How do customer order patterns vary by city or country?**


In [0]:
%sql
-- Customer order patterns by city and country
SELECT 
  c.city,
  c.country,
  COUNT(DISTINCT o.orderid) AS total_orders,
  COUNT(DISTINCT o.customerid) AS total_customers,
  ROUND(SUM(od.total_price), 2) AS total_sales,
  ROUND(AVG(od.total_price), 2) AS avg_order_value,
  ROUND(COUNT(DISTINCT o.orderid) / COUNT(DISTINCT o.customerid), 2) AS avg_orders_per_customer
FROM customers c
JOIN orders o ON c.customerid = o.customerid
JOIN orders_details od ON o.orderid = od.orderid
GROUP BY c.city, c.country
ORDER BY total_sales DESC;

city,country,total_orders,total_customers,total_sales,avg_order_value,avg_orders_per_customer
Cunewalde,Germany,28,1,117477.44,1366.02,28.0
Boise,USA,31,1,115663.79,997.1,31.0
Graz,Austria,30,1,113229.73,1110.1,30.0
Cork,Ireland,19,1,57311.14,1042.02,19.0
Rio de Janeiro,Brazil,34,3,53994.28,650.53,11.33
Albuquerque,USA,18,1,52243.41,735.82,18.0
London,UK,40,4,52099.01,526.25,10.0
null,null,48,8,50958.42,450.96,6.0
São Paulo,Brazil,31,4,45780.62,558.3,7.75
Bräcke,Sweden,19,1,32551.7,723.37,19.0


**Can we cluster customers based on total spend, order count, and preferred categories?**


In [0]:
%sql
-- Cluster customers based on total spend, order count, and preferred categories
SELECT
  c.customerid,
  ROUND(SUM(od.total_price), 2) AS total_spend,
  COUNT(DISTINCT o.orderid) AS order_count,
  FIRST(c.categoryname) AS preferred_category,
  CASE
    WHEN SUM(od.total_price) >= 20000 AND COUNT(DISTINCT o.orderid) >= 10 THEN 'High Value'
    WHEN SUM(od.total_price) >= 10000 AND COUNT(DISTINCT o.orderid) >= 5 THEN 'Medium Value'
    ELSE 'Low Value'
  END AS customer_cluster
FROM customers c
JOIN orders o ON c.customerid = o.customerid
JOIN orders_details od ON o.orderid = od.orderid
JOIN products p ON od.productid = p.productid
JOIN categories c ON p.categoryid = c.categoryid
GROUP BY c.customerid
ORDER BY total_spend DESC;

customerid,total_spend,order_count,preferred_category,customer_cluster
QUICK,117477.44,28,Condiments,High Value
SAVEA,115663.79,31,Seafood,High Value
ERNSH,113229.73,30,Produce,High Value
HUNGO,57311.14,19,Dairy Products,High Value
RATTC,52243.41,18,Confections,High Value
HANAR,34099.0,14,Dairy Products,High Value
FOLKO,32551.7,19,Confections,High Value
MEREP,32202.05,13,Beverages,High Value
KOENE,31744.2,14,Meat/Poultry,High Value
QUEEN,30222.5,13,Confections,High Value


**Which product categories or products contribute most to order revenue?**

In [0]:
%sql
-- Top product categories and products by total order revenue
SELECT
  cat.categoryname,
  p.productname,
  ROUND(SUM(od.total_price), 1) AS total_revenue
FROM orders_details od
JOIN products p ON od.productid = p.productid
JOIN categories cat ON p.categoryid = cat.categoryid
GROUP BY cat.categoryname, p.productname
ORDER BY total_revenue DESC;

categoryname,productname,total_revenue
Beverages,Côte de Blaye,149983.1
Meat/Poultry,Thüringer Rostbratwurst,87734.3
Dairy Products,Raclette Courdavault,76293.4
Dairy Products,Camembert Pierrot,50282.7
Confections,Tarte au sucre,49825.3
Grains/Cereals,Gnocchi di nonna Alice,45118.1
Produce,Manjimup Dried Apples,44740.6
Meat/Poultry,Alice Mutton,35479.9
Seafood,Carnarvon Tigers,31985.4
Produce,Rössle Sauerkraut,26864.4


**Are there any correlations between orders and customer location or product category?**

In [0]:
%sql
-- Correlation between orders and customer location (city/country)
SELECT
  c.city,
  c.country,
  COUNT(DISTINCT o.orderid) AS total_orders,
  ROUND(SUM(od.total_price), 2) AS total_sales,
  ROUND(AVG(od.total_price), 2) AS avg_order_value
FROM customers c
JOIN orders o ON c.customerid = o.customerid
JOIN orders_details od ON o.orderid = od.orderid
GROUP BY c.city, c.country
ORDER BY total_orders DESC;



city,country,total_orders,total_sales,avg_order_value
null,null,48,50958.42,450.96
London,UK,40,52099.01,526.25
Rio de Janeiro,Brazil,34,53994.28,650.53
São Paulo,Brazil,31,45780.62,558.3
Boise,USA,31,115663.79,997.1
Graz,Austria,30,113229.73,1110.1
México D.F.,Mexico,28,24072.45,334.34
Cunewalde,Germany,28,117477.44,1366.02
Cork,Ireland,19,57311.14,1042.02
Bräcke,Sweden,19,32551.7,723.37


In [0]:
%sql
-- Correlation between orders and product category
SELECT
  cat.categoryname,
  COUNT(DISTINCT od.orderid) AS total_orders,
  ROUND(SUM(od.total_price), 2) AS total_sales,
  ROUND(AVG(od.total_price), 2) AS avg_order_value
FROM orders_details od
JOIN products p ON od.productid = p.productid
JOIN categories cat ON p.categoryid = cat.categoryid
GROUP BY cat.categoryname
ORDER BY total_orders DESC;

categoryname,total_orders,total_sales,avg_order_value
Beverages,354,286501.95,709.16
Dairy Products,303,251310.94,686.64
Confections,295,177080.08,530.18
Seafood,291,141603.21,429.1
Condiments,193,113683.38,526.31
Grains/Cereals,182,100717.92,513.87
Meat/Poultry,161,178177.65,1029.93
Produce,129,105262.42,773.99


**How frequently do different customer segments place orders?**


In [0]:
%sql
-- Frequency of orders by customer segment
WITH customer_segments AS (
  SELECT
    c.customerid,
    CASE
      WHEN SUM(od.total_price) >= 20000 AND COUNT(DISTINCT o.orderid) >= 10 THEN 'High Value'
      WHEN SUM(od.total_price) >= 10000 AND COUNT(DISTINCT o.orderid) >= 5 THEN 'Medium Value'
      ELSE 'Low Value'
    END AS customer_segment
  FROM customers c
  JOIN orders o ON c.customerid = o.customerid
  JOIN orders_details od ON o.orderid = od.orderid
  GROUP BY c.customerid
)
SELECT
  cs.customer_segment,
  COUNT(DISTINCT o.orderid) AS total_orders,
  COUNT(DISTINCT cs.customerid) AS total_customers,
  ROUND(COUNT(DISTINCT o.orderid) / COUNT(DISTINCT cs.customerid), 2) AS avg_orders_per_customer
FROM customer_segments cs
JOIN orders o ON cs.customerid = o.customerid
GROUP BY cs.customer_segment
ORDER BY total_orders DESC;

customer_segment,total_orders,total_customers,avg_orders_per_customer
High Value,342,20,17.1
Low Value,273,48,5.69
Medium Value,215,21,10.24


**What is the geographic and title-wise distribution of employees?**


In [0]:
%sql
SELECT 
  city,
  country,
  title,
  COUNT(*) AS employee_count
FROM employees
GROUP BY city, country, title
ORDER BY employee_count DESC;

city,country,title,employee_count
London,UK,Sales Representative,3
London,UK,Sales Manager,1
Seattle,USA,Sales Representative,1
Kirkland,USA,Sales Representative,1
Redmond,USA,Sales Representative,1
Tacoma,USA,"Vice President, Sales",1
Seattle,USA,Inside Sales Coordinator,1


**What trends can we observe in hire dates across employee titles?**


In [0]:
%sql
-- Trends in hire dates across employee titles
SELECT
  title,
  MIN(hiredate) AS earliest_hire_date,
  MAX(hiredate) AS latest_hire_date,
  COUNT(*) AS employee_count,
  ROUND(AVG(YEAR(hiredate)), 2) AS avg_hire_year
FROM employees
GROUP BY title
ORDER BY earliest_hire_date ASC;

title,earliest_hire_date,latest_hire_date,employee_count,avg_hire_year
Sales Representative,1992-04-01,1994-11-15,6,1993.0
"Vice President, Sales",1992-08-14,1992-08-14,1,1992.0
Sales Manager,1993-10-17,1993-10-17,1,1993.0
Inside Sales Coordinator,1994-03-05,1994-03-05,1,1994.0


**What patterns exist in employee title and courtesy title distributions?**


In [0]:
%sql
-- Distribution of employee titles and courtesy titles
SELECT
  title,
  titleofcourtesy,
  COUNT(*) AS employee_count
FROM employees
GROUP BY title, titleofcourtesy
ORDER BY employee_count DESC;

title,titleofcourtesy,employee_count
Sales Representative,Ms.,3
Sales Representative,Mr.,2
Sales Manager,Mr.,1
Sales Representative,Mrs.,1
"Vice President, Sales",Dr.,1
Inside Sales Coordinator,Ms.,1


**Are there correlations between product pricing, stock levels, and sales performance?**


In [0]:
%sql
-- Correlation between product pricing, stock levels, and sales performance
SELECT
  p.productid,
  p.productname,
  p.unitprice,
  p.unitsinstock,
  ROUND(SUM(od.total_price), 2) AS total_sales,
  COUNT(DISTINCT od.orderid) AS total_orders,
  ROUND(AVG(od.total_price), 2) AS avg_order_value
FROM products p
JOIN orders_details od ON p.productid = od.productid
GROUP BY p.productid, p.productname, p.unitprice, p.unitsinstock
ORDER BY total_sales DESC;

productid,productname,unitprice,unitsinstock,total_sales,total_orders,avg_order_value
38,Côte de Blaye,263.5,17,149983.1,24,6249.3
29,Thüringer Rostbratwurst,123.79,0,87734.35,32,2741.7
59,Raclette Courdavault,55.0,79,76293.45,54,1412.84
60,Camembert Pierrot,34.0,19,50282.74,51,985.94
62,Tarte au sucre,49.3,17,49825.3,48,1038.03
56,Gnocchi di nonna Alice,38.0,21,45118.1,50,902.36
51,Manjimup Dried Apples,53.0,20,44740.6,39,1147.19
17,Alice Mutton,39.0,0,35479.9,37,958.92
18,Carnarvon Tigers,62.5,42,31985.35,27,1184.64
28,Rössle Sauerkraut,45.6,26,26864.35,33,814.07


**How does product demand change over months or seasons?**


In [0]:
%sql
-- Product demand trends by month and season
SELECT
  p.productid,
  p.productname,
  MONTH(o.orderdate) AS order_month,
  CASE
    WHEN MONTH(o.orderdate) IN (12, 1, 2) THEN 'Winter'
    WHEN MONTH(o.orderdate) IN (3, 4, 5) THEN 'Spring'
    WHEN MONTH(o.orderdate) IN (6, 7, 8) THEN 'Summer'
    ELSE 'Fall'
  END AS order_season,
  COUNT(od.orderid) AS total_orders,
  SUM(od.quantity) AS total_quantity,
  ROUND(SUM(od.total_price), 2) AS total_sales
FROM orders_details od
JOIN orders o ON od.orderid = o.orderid
JOIN products p ON od.productid = p.productid
GROUP BY p.productid, p.productname, MONTH(o.orderdate),
         CASE
           WHEN MONTH(o.orderdate) IN (12, 1, 2) THEN 'Winter'
           WHEN MONTH(o.orderdate) IN (3, 4, 5) THEN 'Spring'
           WHEN MONTH(o.orderdate) IN (6, 7, 8) THEN 'Summer'
           ELSE 'Fall'
         END
ORDER BY p.productname, order_month;

productid,productname,order_month,order_season,total_orders,total_quantity,total_sales
17,Alice Mutton,1,Winter,3,108,4211.5
17,Alice Mutton,2,Winter,2,8,296.4
17,Alice Mutton,3,Spring,8,200,6980.5
17,Alice Mutton,4,Spring,3,27,1053.0
17,Alice Mutton,5,Spring,2,89,3470.9
17,Alice Mutton,6,Summer,3,73,2846.8
17,Alice Mutton,7,Summer,2,34,1325.95
17,Alice Mutton,8,Summer,2,130,4836.0
17,Alice Mutton,9,Fall,2,30,935.75
17,Alice Mutton,10,Fall,2,70,2417.8


**Can we identify anomalies in product sales or revenue performance?**


In [0]:
%sql
-- Identify anomalies in product sales or revenue performance using Z-score
WITH product_sales_stats AS (
  SELECT
    p.productid,
    p.productname,
    ROUND(SUM(od.total_price), 2) AS total_sales,
    COUNT(od.orderid) AS total_orders
  FROM products p
  JOIN orders_details od ON p.productid = od.productid
  GROUP BY p.productid, p.productname
),
sales_summary AS (
  SELECT
    AVG(total_sales) AS avg_sales,
    STDDEV(total_sales) AS stddev_sales,
    AVG(total_orders) AS avg_orders,
    STDDEV(total_orders) AS stddev_orders
  FROM product_sales_stats
)
SELECT
  ps.productid,
  ps.productname,
  ps.total_sales,
  ps.total_orders,
  ROUND((ps.total_sales - ss.avg_sales) / ss.stddev_sales, 2) AS sales_zscore,
  ROUND((ps.total_orders - ss.avg_orders) / ss.stddev_orders, 2) AS orders_zscore
FROM product_sales_stats ps
CROSS JOIN sales_summary ss
WHERE ABS((ps.total_sales - ss.avg_sales) / ss.stddev_sales) > 2
   OR ABS((ps.total_orders - ss.avg_orders) / ss.stddev_orders) > 2
ORDER BY sales_zscore DESC, orders_zscore DESC;

productid,productname,total_sales,total_orders,sales_zscore,orders_zscore
38,Côte de Blaye,149983.1,24,6.09,-0.31
29,Thüringer Rostbratwurst,87734.35,32,3.23,0.31
59,Raclette Courdavault,76293.45,54,2.7,2.01


**Are there any regional trends in supplier distribution and pricing?**


In [0]:
%sql
-- Regional trends in supplier distribution and pricing
SELECT
  s.country,
  s.city,
  COUNT(DISTINCT s.supplierid) AS supplier_count,
  ROUND(AVG(p.unitprice), 2) AS avg_product_price,
  ROUND(MIN(p.unitprice), 2) AS min_product_price,
  ROUND(MAX(p.unitprice), 2) AS max_product_price
FROM suppliers s
JOIN products p ON s.supplierid = p.supplierid
GROUP BY s.country, s.city
ORDER BY supplier_count DESC, avg_product_price DESC;

country,city,supplier_count,avg_product_price,min_product_price,max_product_price
null,null,9,26.41,6.0,97.0
France,Paris,1,140.75,18.0,263.5
Germany,Frankfurt,1,44.68,7.75,123.79
Canada,Ste-Hyacinthe,1,38.9,28.5,49.3
null,France,1,31.67,25.0,40.0
USA,Ann Arbor,1,31.67,25.0,40.0
Germany,Berlin,1,29.71,14.0,43.9
Spain,Oviedo,1,29.5,21.0,38.0
Italy,Salerno,1,28.75,19.5,38.0
UK,Manchester,1,28.18,9.2,81.0


**How are suppliers distributed across different product categories?**


In [0]:
%sql
-- Distribution of suppliers across product categories
SELECT
  cat.categoryname,
  COUNT(DISTINCT s.supplierid) AS supplier_count
FROM products p
JOIN categories cat ON p.categoryid = cat.categoryid
JOIN suppliers s ON p.supplierid = s.supplierid
GROUP BY cat.categoryname
ORDER BY supplier_count DESC;

categoryname,supplier_count
Condiments,8
Seafood,8
Beverages,8
Confections,6
Grains/Cereals,5
Meat/Poultry,5
Produce,5
Dairy Products,4


**How do supplier pricing and categories relate across different regions?**


In [0]:
%sql
use workspace.northwind_sales_analysis;

-- Supplier pricing and category relationships across regions
SELECT
  s.City,
  s.Country,
  cat.CategoryName,
  ROUND(AVG(p.UnitPrice), 2) AS avg_product_price,
  ROUND(MIN(p.UnitPrice), 2) AS min_product_price,
  ROUND(MAX(p.UnitPrice), 2) AS max_product_price,
  COUNT(DISTINCT s.SupplierID) AS supplier_count
FROM workspace.northwind_sales_analysis.suppliers s
JOIN products p ON s.SupplierID = p.SupplierID
JOIN categories cat ON p.CategoryID = cat.CategoryID
GROUP BY s.Country, s.City, cat.CategoryName
ORDER BY s.Country, s.City, avg_product_price DESC;

City,Country,CategoryName,avg_product_price,min_product_price,max_product_price,supplier_count
null,null,Meat/Poultry,56.27,32.8,97.0,3
null,null,Dairy Products,44.5,34.0,55.0,1
null,null,Condiments,29.7,15.5,43.9,2
null,null,Produce,28.75,10.0,53.0,3
null,null,Seafood,21.29,6.0,62.5,5
null,null,Beverages,15.25,14.0,18.0,2
null,null,Confections,13.23,9.5,17.45,2
null,null,Grains/Cereals,7.0,7.0,7.0,1
France,null,Condiments,32.5,25.0,40.0,1
France,null,Produce,30.0,30.0,30.0,1
